# Od klávesnice do neuronu · podívej se dovnitř drobka

Drobek je malý model, co umí psát kód (Qwen2.5-Coder, 1,5 miliardy čísel). Běží ti celý na GPU zdarma.
V přednášce jsme se mu dívali až na jeden neuron. Tady si to zkusíš sám: každá buňka něco spočítá a ukáže,
a drobek řekne, co z toho plyne. Nahoře dej **Prostředí → Změnit typ → GPU (T4)**, pak **Prostředí → Spustit vše**.
Trvá to pár minut, většinu zabere stažení modelu. Pak už si hraj: posuvníky znovu pouštěj.

In [ ]:
# ── Příprava: stáhneme kurz a probudíme drobka ──
# (repo je zatím soukromé, tak se bere zip nahraný do /content; až bude veřejné, nahradí ho git clone)
import os
if not os.path.exists('od-klavesnice-do-neuronu-public'):
    !unzip -q -o /content/kurz.zip -d /content 2>/dev/null || git clone -q https://github.com/moudrkat/od-klavesnice-do-neuronu.git od-klavesnice-do-neuronu-public
%cd od-klavesnice-do-neuronu-public/dovnitr
!pip install -q transformers
import kurz                         # celý kurz je v jednom souboru vedle: kurz.py
print(kurz.load())                  # načte drobka na GPU — stane se jednou, pak už jen voláme funkce
kurz.drobek('Ahoj! Jsem drobek. Tyhle čísla jsem já celý. Pojď se podívat, co v nich je.')

## Lekce 2 · jak malý drobek je

Celý model je jen hromada čísel ve třech kapsách: slovník, neurony, hlavy. Žádná magie navíc.

In [ ]:
# kurz.fakta() rozkreslí, kolik čísel je ve slovníku, v neuronech a v hlavách,
# a změří, kolik slov drobek napíše za vteřinu.
rychlost, ukazka = kurz.fakta()
print(f'{rychlost} slov za vteřinu, a tohle zrovna napsal:')
print(ukazka)
kurz.drobek(f'Mám přes půtora miliardy čísel, ale taky umím psát kód rychlostí {rychlost} slov/s. Malej, ale šikovnej.')

## Lekce 3 · co vlastně vejde dovnitř

Tvůj prompt se rozseká na tokeny (kousky slov). Každý dostane barvu podle toho, odkud je.

In [ ]:
# kurz.prompt_pas() nakreslí pás tokenů obarvený podle částí promptu
# (šedě první token, okrově systémový prompt, modře tvoje zadání, tmavě jeho kód).
pocty = kurz.prompt_pas()
print('kolik tokenů z které části:', pocty)
kurz.drobek('Dovnitř nejde text, ale čísla. A tvoje zadání je jen pár tokenů z celku — zbytek je šablona a kód.')

## Lekce 4 · jak ze čtení vznikne další slovo

Model si pod posledním tokenem (`return`) nese 1 536 čísel — svou pracovní paměť. Každá vrstva do nich něco připíše.
Kdybychom se zastavili uprostřed, jaké slovo by řekl?

In [ ]:
# kurz.vrstvami() vezme proud po každé z 28 vrstev a přečte, které slovo by vyhrálo.
# Graf ukáže, jak skóre slov sum / count / len rostou vrstvu po vrstvě.
slova, prelom = kurz.vrstvami()
print(f'Až do vrstvy {prelom} by řekl něco jiného; teprve tam vyhraje sum.')
kurz.drobek(f'Koukej: dlouho to vypadá na count, a pak ve vrstvě {prelom} to přeskočí na sum. Rozmyslím si to až nahoře.')

### A teď si pohni teplotou

Teplota je jediné číslo, kterým se skóre vydělí, než z nich udělá pravděpodobnosti. 0 = vždycky nejjistější slovo. Čím výš, tím divočejš. Posuň a spusť znovu.

In [ ]:
teplota = 1 #@param {type:"slider", min:0, max:3, step:0.5}
# kurz.teplota(T) ukáže sloupce pravděpodobností dalšího slova a jeden vzorek kódu při téhle teplotě.
slova, vzorek = kurz.teplota(teplota)
print('napsal by:', vzorek[:160])
kurz.drobek('Na nule jsem spolehlivej. Vytoč mě na dvojku a začnu si vymýšlet slova z pěti jazyků najednou.', face='drobek_prekvapeny')

## Lekce 5 · dva ovladače ve vrstvě 20

Ve vrstvě 20 je jedna hlava (20·6), co čte tvoje zadání, a jeden neuron (20·3545), co rozhodne, jakou funkci napsat.
Nejdřív: na co se ta hlava z pozice `return` vlastně dívá?

In [ ]:
# kurz.kam_se_diva(hlava) nakreslí, kam se z pozice return dívá daná hlava vrstvy 20.
# Sloupce jsou obarvené podle částí promptu — uvidíš, že 20·6 míří na tvoje zadání.
kam = kurz.kam_se_diva(6)
print('nejvíc se dívá na:', kam[:3])
kurz.drobek('Tahle hlava se z místa, kde se rozhoduju, dívá přímo na tvoje slovo "list". Čte zadání.')

### Zahýbej hlavou a neuronem

Zesil nebo zhasni hlavu 20·6 a neuron 20·3545 a koukej, co drobek napíše **za** `return`. Jeden neuron, tři různé programy.

In [ ]:
hlava = 1 #@param {type:"slider", min:0, max:4, step:1}
neuron = 1 #@param {type:"slider", min:0, max:4, step:1}
# kurz.paka(hlava, neuron) vynásobí hlavu a neuron tvým číslem a nechá drobka dopsat řádek za return.
pravd, za_return = kurz.paka(hlava, neuron)
print('pravděpodobnosti dalšího slova:', pravd)
print('za return napíše:', za_return)
kurz.drobek('Zhasni neuron a napíšu len. Zesil ho čtyřikrát a napíšu tasks.count(True). Hlava je jen přívod — ta kód nezmění.')

### Který neuron to rozhodl?

V přednášce jsem vypnula všech 8 960 neuronů vrstvy 20 jeden po druhém. Jen jeden hnul výsledkem. Obrázek je z té zkoušky; živě si ověř tři z nich.

In [ ]:
# kurz.sken_obrazek() ukáže všech 8 960 neuronů (hotová data) — jeden trčí dolů, to je 3545.
nej = kurz.sken_obrazek()
# kurz.zhasni_neuron(n) vypne JEN jeden neuron a změří, o kolik spadne jistota slova sum (živě, vteřina).
print('3545:', kurz.zhasni_neuron(3545), '← (základ, po vypnutí, pokles)')
print('1507:', kurz.zhasni_neuron(1507), '← tenhle taky svítí, ale skoro nic neudělá')
muj = 100 #@param {type:"integer"}
print(f'{muj}:', kurz.zhasni_neuron(muj), '← zkus libovolný z 0–8959')
kurz.drobek('Ze všech 8 960 neuronů hne výsledkem jen 3545. Většina lidí má jeden takovej. Já taky.', face='senior')

## Lekce 6 · přičti směr, dostaneš jiný jazyk

Spočítám směr „piš JavaScript místo Pythonu“ a přičtu ho do proudu ve středních vrstvách. Zadání o jazyce neřeknu ani slovo.

In [ ]:
sila = 6 #@param {type:"slider", min:0, max:8, step:1}
# kurz.smer(sila) přičte do proudu směr k JavaScriptu a nechá drobka napsat funkci na faktoriál.
# sila 0 = Python, kolem 6 přeskočí do JavaScriptu, moc vysoko = kaše.
print(kurz.smer(sila))
kurz.drobek('Nikdo mi neřekl "piš JavaScript". Jen jsem dostal šťouchanec správným směrem — a přepnul jazyk.', face='drobek_zmateny')

## Lekce 7 · umí drobek vůbec psát kód?

Standardní test je 164 úloh (drobek jich zvládne 110). Celé to trvá deset minut a je to nuda, tak si pustíme jen tři náhodné a spustíme je naostro.

In [ ]:
# kurz.humaneval_ukazka(3) vybere tři náhodné úlohy, drobek je dořeší a my je OPRAVDU spustíme (✓/✗).
vysledky, cely = kurz.humaneval_ukazka(3)
for r in vysledky: print('✓' if r['ok'] else '✗', r['task'])
print(f'Na celém testu: {cely}/164.')
kurz.drobek('Tři ze tří? Nebo jsem jednu zbabral? Na celých 164 jich dám 110. Na domácí GPU to není špatný.')

## Bonus · parta (volitelné, stáhne i model 4B, ~8 GB)

Drobek sám občas chybuje. V přednášce jsem mu dala parťáka: větší model napíše zadání, drobek kód, a třetí ho zkontroluje.

In [ ]:
# Pozor, tohle stáhne druhý, větší model (~8 GB) a chvíli to trvá.
!python orchestrate.py '["Make me a to-do app so I can see how many tasks are done"]'
kurz.drobek('Refiner mi napsal zadání, já kód, reviewer ho proklepl. Sám bych to nedal — a přesně tím to celé začalo.', face='partak_reviewer')

---
Chceš víc? Celý kurz, interaktivní vrstva 20 a všechna data jsou na **github.com/moudrkat/od-klavesnice-do-neuronu**.
Všechno ostatní na **github.com/moudrkat**.